# Quantizing Qwen Models with TurboQuant-v3

This notebook demonstrates how to apply TurboQuant-v3's INT4 + AWQ + Protected Channels quantization to a Qwen model (e.g., Qwen2.5-7B). You'll see:

- Loading a Qwen model from HuggingFace.
- Configuring quantization parameters.
- Quantizing and saving the compressed model.
- Running inference and comparing memory usage/speed.

## 1. Install Dependencies

Make sure you have TurboQuant-v3 installed from source:

```bash
git clone https://github.com/Kubenew/TurboQuant-v3.git
cd TurboQuant-v3
pip install -e ".[dev]"
```

If you have CUDA and want high-performance kernels:

```bash
python setup_cuda.py install
```

## 2. Import Libraries

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from turboquant import TurboQuantConfig, quantize_model, save_quantized_model, load_quantized_model

## 3. Load Original Qwen Model

In [ ]:
model_name = "Qwen/Qwen2.5-7B"  # or "Qwen/Qwen2-7B", "Qwen/Qwen2.5-14B", etc.
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load in FP16 on GPU (adjust device_map if needed)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,   # needed for Qwen
)

## 4. Configure Quantization

In [ ]:
quant_config = TurboQuantConfig(
    bits=4,
    group_size=128,
    version="gemm",               # use "gemm" (default) or "exllama" for faster inference
    zero_point=True,
    activation_aware=True,        # AWQ-style scaling
    outlier_keep_ratio=0.02,      # keep 2% of channels in FP16
    rank=8,                       # low-rank SVD correction
)

## 5. Quantize the Model

In [ ]:
# This replaces linear layers with QuantizedLinear
quantized_model = quantize_model(model, quantization_config=quant_config)

# Move to GPU if not already
quantized_model = quantized_model.to("cuda")

## 6. Save Quantized Model

In [ ]:
save_quantized_model(
    quantized_model,
    "./qwen-7b-turboquant",
    quant_config,
    save_tokenizer=True,
)

## 7. Load Quantized Model Later

In [ ]:
loaded_model = load_quantized_model(
    "./qwen-7b-turboquant",
    device_map="auto",
)

## 8. Run Inference

In [ ]:
prompt = "Explain the concept of quantization in large language models."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = loaded_model.generate(**inputs, max_new_tokens=128)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 9. Measure Memory Savings

In [ ]:
def model_memory_mb(model):
    mem = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 * 1024)
    return mem

print(f"Original model memory (estimated): {model_memory_mb(model):.1f} MB")
print(f"Quantized model memory (estimated): {model_memory_mb(quantized_model):.1f} MB")

> **Expected:** ~4× reduction (~14 GB → ~3.5 GB for 7B model).

## 10. Performance Comparison

In [ ]:
import time

def measure_inference(model, inputs, repeats=10):
    model.eval()
    with torch.no_grad():
        # Warmup
        for _ in range(3):
            _ = model(**inputs)
        torch.cuda.synchronize()
        start = time.time()
        for _ in range(repeats):
            _ = model(**inputs)
        torch.cuda.synchronize()
        elapsed = time.time() - start
    return elapsed / repeats

# Original model (still in memory)
orig_time = measure_inference(model, inputs)
quant_time = measure_inference(quantized_model, inputs)

print(f"Original inference time per step: {orig_time*1000:.2f} ms")
print(f"Quantized inference time per step: {quant_time*1000:.2f} ms")
print(f"Speedup: {orig_time/quant_time:.2f}x")

## 11. Verify Quality

Check perplexity on a small validation set or compare generated outputs. The `compute_metrics` function from `turboquant` can be used on weights, but for end-to-end quality, run a few generations.

In [ ]:
# Example: compare a few generations
test_prompts = [
    "The future of AI is",
    "To solve climate change, we must",
    "In three sentences, explain relativity."
]

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out_orig = model.generate(**inputs, max_new_tokens=50)
        out_quant = quantized_model.generate(**inputs, max_new_tokens=50)
    print("Original:", tokenizer.decode(out_orig[0], skip_special_tokens=True))
    print("Quantized:", tokenizer.decode(out_quant[0], skip_special_tokens=True))
    print("-" * 80)

## 12. Notes for Qwen-Specific Architecture

- Qwen models use **rotary position embeddings** and **GQA** (grouped-query attention). TurboQuant-v3 quantizes only linear layers, leaving attention mechanisms untouched. This should work out-of-the-box.
- For **Qwen2.5** and later, `trust_remote_code=True` is required. The quantization wrapper automatically handles `q_proj`, `k_proj`, `v_proj`, `o_proj`, and MLP layers.
- If you encounter `KeyError` for a layer not covered, check `src/turboquant/hf_modules.py` and add it if needed.

## 13. Using CUDA Kernels (Optional)

For best performance, compile the CUDA extension and use `version="exllama"`:

```python
quant_config = TurboQuantConfig(..., version="exllama")
```

Then ensure you have compiled:

```bash
python setup_cuda.py install
```

The quantized model will automatically use the faster kernels when available.

## 14. Troubleshooting

- **Out of memory?** Reduce `group_size` (e.g., 64) or `rank` (0 to disable SVD).
- **Slow inference?** Use `version="exllama"` and make sure CUDA kernels are installed.
- **Quality drop?** Increase `outlier_keep_ratio` (e.g., 0.05) or `rank` (e.g., 16).

## Next Steps

- Try on larger Qwen models (14B, 32B) – the compression ratio remains ~4×.
- Experiment with 8-bit quantization for higher quality (`bits=8`).
- Contribute improvements to the CUDA kernels for even faster inference.

**Happy Quantizing!**